# Layout fine-tuning with the toolchain in the loop

The gdsfactory-lane 5T-OTA generator ([`gen_5t_ota_gf.py`](gen_5t_ota_gf.py)) exposes its
free layout constants as a `LayoutParams` dataclass. That makes the layout **optimizable
against the real toolchain**: every candidate is generated, checked with the PDK's own
DRC + LVS decks (hard gate), parasitic-extracted with klayout-pex, and simulated with
ngspice on the `amp_001_5t` open-loop AC bench. The score (lower = better):

$$\mathrm{score} = \frac{\mathrm{area}}{\mathrm{area}_0} + \frac{\mathrm{UGF\ loss}}{\mathrm{loss}_0}$$

normalized to the original hand-tuned layout (232.1 µm², 0.726 MHz loss → score 2.0).
This notebook walks the loop: one evaluation, a short live search, and the analysis of a
full run. It is the deterministic core of the **calibrate-by-feedback loop** in the
meta-repo's `doc/plan_layout_automation.md` — A similar tools an LLM layout agent will drive.

Requirements: `ai_env` (gdsfactory + ihp-gdsfactory + nevergrad), the `pex` conda env
(klayout-pex), `PDK_ROOT`, and the batch KLayout on `PATH` — see [`../INSTALL.md`](../INSTALL.md).


In [ ]:
import json
import os

os.environ.setdefault("PDK_ROOT", os.path.expanduser("~/local/pdks"))

from optimize_layout import AREA0, BOUNDS, LOSS0, PEX_BIN, evaluate, render_png
from gen_5t_ota_gf import LayoutParams

os.environ["PATH"] = PEX_BIN + os.pathsep + os.environ["PATH"]  # kpex
print("search space:")
for k, (lo, hi) in BOUNDS.items():
    print(f"  {k:<8} [{lo}, {hi}]  (current default {getattr(LayoutParams(), k)})")

## 1. One evaluation

`evaluate()` runs the full pipeline for one parameter set and returns a structured row —
exactly what a search algorithm (or an agent) needs to decide its next move. The committed
defaults are the optimizer's winner, so this should come back DRC/LVS-clean with
area ≈ 205.9 µm². (~25 s: generate + DRC + LVS + kpex + ngspice.)


In [ ]:
import sys
from sim_pex_compare import run_ac

WORK = "opt_nb"
os.makedirs(WORK, exist_ok=True)
_, PRE_UGF, _ = run_ac(os.path.abspath("ota_5t_gf_lvs.spice"), "ota_5t_gf", "pre", WORK)
print(f"pre-layout UGF (schematic reference): {PRE_UGF/1e6:.3f} MHz")

row = evaluate(LayoutParams(), os.path.join(WORK, "default"), PRE_UGF)
row

In [ ]:
from IPython.display import Image, display

display(Image(os.path.join(WORK, "default", "layout.png"), width=640))

## 2. A short search, live

The real runs used `python optimize_layout.py --budget 30` (twice — see §3). Here a
6-trial `TwoPointsDE` search shows the loop turning over: candidates that violate DRC or
LVS are penalized (that is how the optimizer locates the legal boundary), the rest get
their PEX + sim metric. Each trial dir keeps its GDS and a `layout.png` for visual
inspection (the CLI equivalent of `--keep-all`).


In [ ]:
import nevergrad as ng

space = ng.p.Instrumentation(**{
    k: ng.p.Scalar(init=getattr(LayoutParams(), k), lower=lo, upper=hi)
    for k, (lo, hi) in BOUNDS.items()})
opt = ng.optimizers.TwoPointsDE(parametrization=space, budget=6)
opt.suggest(**{k: getattr(LayoutParams(), k) for k in BOUNDS})   # seed the known-good point

rows = []
for i in range(6):
    cand = opt.ask()
    p = LayoutParams(**{k: round(v, 2) for k, v in cand.kwargs.items()})  # 0.01 µm grid
    r = evaluate(p, os.path.join(WORK, f"t{i:02d}"), PRE_UGF)
    r["trial"] = i
    opt.tell(cand, float(r["score"]))
    rows.append(r)
    print(f"[{i}] {r['status']:<12} area={r.get('area_um2','-'):>6} "
          f"loss={r.get('ugf_loss_mhz','-'):>6} score={r['score']}")

## 3. Analyzing a full run

Every trial is logged as one JSONL row (`opt_out/trials.jsonl` + `best.json`). Below: the
6 trials from this notebook in the area-vs-error plane. In the real 2×30 run the winner
reached **205.9 µm² (−11.3 %) with 0.699 MHz loss** — mostly by compacting the row
channels (`ch_y` 1.8 → 0.92 µm) while *widening* `gap_x`/`vdd_off` to protect the error
term; those values are now the committed `LayoutParams` defaults.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame([{**r["params"], **{k: v for k, v in r.items() if k != "params"}}
                   for r in rows])
ok = df[df.status == "ok"]
bad = df[df.status != "ok"]
fig, ax = plt.subplots(figsize=(7, 4.5))
if len(ok):
    ax.scatter(ok.area_um2, ok.ugf_loss_mhz, c=ok.score, cmap="viridis_r", s=60,
               zorder=3, label="feasible (DRC+LVS clean)")
if len(bad):
    y_bad = (ok.ugf_loss_mhz.max() if len(ok) else LOSS0)
    ax.scatter(bad.area_um2, [y_bad] * len(bad), marker="x", c="crimson", s=60,
               zorder=3, label=f"rejected ({', '.join(bad.status.unique())})")
ax.scatter([AREA0], [LOSS0], marker="*", s=220, c="orange", zorder=4,
           label="hand-tuned baseline")
ax.set_xlabel("area (µm²)")
ax.set_ylabel("post-layout UGF loss (MHz)")
ax.set_title("layout candidates: area vs parasitic error")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

df[[c for c in df.columns if c not in ("drc", "lvs")]]

## 4. Interactive optimization trace (Plotly)

Interactive views of the **full saved runs** (`opt_out*/trials.jsonl` — the 30-trial CC run
and the earlier RC run), in the same spirit as the `spicexplorer` optimizer's Plotly
reports (`spicexplorer.viz.Optimization_Log_Visualizer`). This notebook stays a lane
prototype — folding layout trials into that package's log/report machinery is future work
(see the meta-repo's `doc/plan_layout_automation.md`).
Hover any point for the full parameter set of that candidate.


In [ ]:
import glob

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"   # interactive in Jupyter/VS Code (CDN js)

runs = {}
for path in sorted(glob.glob("opt_out*/trials.jsonl")):
    tag = path.split("/")[0].replace("opt_out", "run") or "run"
    runs[tag] = pd.DataFrame([{**json.loads(l)["params"],
                               **{k: v for k, v in json.loads(l).items() if k != "params"}}
                              for l in open(path)])
if not runs:                       # fresh clone: fall back to this notebook's demo rows
    runs = {"demo": df}
for tag, d in runs.items():
    d["trial"] = d.get("trial", pd.Series(range(len(d)))).fillna(pd.Series(range(len(d))))
    print(tag, "->", len(d), "trials,", (d.status == "ok").sum(), "feasible")

In [ ]:
HOVER = ["gap_x", "ch_y", "edge_x", "w_m1", "w_m2", "tab_w", "ib_off", "vdd_off", "vss_off"]

fig = go.Figure()
for tag, d in runs.items():
    d = d.sort_values("trial")
    feas = d[d.status == "ok"]
    fig.add_trace(go.Scatter(
        x=feas.trial, y=feas.score, mode="markers", name=f"{tag}: feasible",
        customdata=feas[HOVER + ["area_um2", "ugf_loss_mhz"]],
        hovertemplate=("trial %{x} score %{y:.3f}<br>area %{customdata[9]:.1f} µm² "
                       "loss %{customdata[10]:.3f} MHz<br>" +
                       "<br>".join(f"{k}=%{{customdata[{i}]:.2f}}" for i, k in enumerate(HOVER)))))
    run_best = d.score.where(d.status == "ok").cummin().ffill()
    fig.add_trace(go.Scatter(x=d.trial, y=run_best, mode="lines",
                             name=f"{tag}: best so far", line=dict(width=2)))
fig.add_hline(y=2.0, line_dash="dot", annotation_text="hand-tuned baseline (score 2.0)")
fig.update_layout(title="Optimization trace — score per trial (lower is better)",
                  xaxis_title="trial", yaxis_title="score", yaxis_range=[1.7, 2.3],
                  height=440, legend=dict(orientation="h", y=-0.2))
fig.show()

In [ ]:
cc = runs.get("run", next(iter(runs.values())))
feas = cc[cc.status == "ok"]
fig = px.scatter(feas, x="area_um2", y="ugf_loss_mhz", color="score",
                 color_continuous_scale="viridis_r", hover_data=HOVER,
                 title="Feasible candidates — area vs parasitic error (CC run)",
                 labels={"area_um2": "area (µm²)", "ugf_loss_mhz": "UGF loss (MHz)"})
fig.add_trace(go.Scatter(x=[AREA0], y=[LOSS0], mode="markers+text", text=["baseline"],
                         textposition="top center", marker=dict(symbol="star", size=16,
                         color="orange"), name="hand-tuned"))
fig.update_layout(height=440)
fig.show()

In [ ]:
fig = px.parallel_coordinates(
    feas, dimensions=HOVER + ["area_um2", "score"], color="score",
    color_continuous_scale="viridis_r",
    title="Which knobs matter — parameters vs outcome (drag axes to filter)")
fig.update_layout(height=420)
fig.show()

## Notes / gotchas (hard-won, also in README.md)

- **In-loop PEX is `--mode CC`** (capacitances only): validated loss-identical to RC on
  this block, and immune to kpex's 2.5D R-mesh occasionally leaving a pin node dangling
  on the tiny gate nets → singular ngspice matrix. Final signoff still uses RC.
- The PEX-netlist operating point wants the `.nodeset` bias guess that
  `sim_pex_compare.py` bakes into the bench deck.
- Searched dimensions are snapped to a **0.01 µm grid** (gdsfactory port widths must be
  an even number of DBU).
- Rejected candidates are informative: `Cnt.g1` (tap contacts too close to devices at
  tight `edge_x`) and `M1.a` (metal slivers at minimal `w_m1`) mark the legal boundary.
- Next step per the plan: joint **sizing + layout** search, and an LLM agent driving
  these same tools with a design-rule knowledge base.
